# End-to-End Pipeline Deployment
**PyGeoVision v2.0 — Notebook 22**

## Real-World Problem
A government agency needs to run weekly automated building extraction across
their entire country — searching, downloading, processing, and exporting
GeoJSON updates without any manual intervention.

## Learning Objectives
1. Build pipelines programmatically and from YAML
2. Schedule with cron expressions
3. Monitor execution and handle failures
4. Deploy models to cloud (AWS/Azure/GCP) and edge (Jetson)
5. Set up production monitoring

```bash
pip install "pygeovision[geo,train,serve]" yaml
```

In [ ]:
import pygeovision as pgv
import yaml, json
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/22_pipeline_deployment")
BASE_DIR.mkdir(parents=True, exist_ok=True)

client = pgv.PyGeoVision()
print(client)
print("\nGoal: Weekly automated building extraction pipeline")
print("Coverage: National scale  |  Fully automated  |  GeoJSON output")

## Step 1: Programmatic Pipeline

In [ ]:
# Build pipeline using the fluent API
p = client.create_pipeline(
    "national_building_extraction",
    description="Weekly automated building footprint update — national scale",
)

p.search(
    bbox=(-74.10, 40.60, -73.70, 40.90),   # Demo: New York City
    providers=["planetary_computer"],
    cloud_cover_max=10,
)
p.download(
    output_dir=str(BASE_DIR/"scenes"),
    post_process=["reproject:EPSG:32618","cog"],
    parallel=4,
)
p.ai_step("segment_buildings", model="segformer-b2", num_classes=2)
p.export(format="geojson", output_dir=str(BASE_DIR/"results"))
p.schedule(cron="0 3 * * 1")   # Every Monday at 03:00 UTC

print("Pipeline created:")
print(p)
print()
print("Pipeline steps:")
for step in p._steps if hasattr(p,'_steps') else []:
    print(f"  {step}")

## Step 2: YAML Pipeline Configuration

In [ ]:
# Production YAML pipeline configuration
pipeline_config = {
    "name"       : "building_extraction_weekly",
    "description": "Automated weekly building footprint update",
    "version"    : "2.0",
    "schedule"   : "0 3 * * 1",   # Every Monday 03:00 UTC
    "defaults"   : {
        "cloud_cover_max": 10,
        "output_dir"     : "./output/buildings/",
    },
    "steps": [
        {
            "name"  : "search",
            "action": "search",
            "params": {
                "bbox"           : [-74.10, 40.60, -73.70, 40.90],
                "providers"      : ["planetary_computer"],
                "date_range"     : ["{{last_week}}", "{{today}}"],
                "cloud_cover_max": 10,
            }
        },
        {
            "name"       : "download",
            "action"     : "download",
            "depends_on" : ["search"],
            "retry_on_fail": 3,
            "params"     : {
                "output_dir" : "./data/scenes/",
                "parallel"   : 4,
                "post_process": ["reproject:EPSG:32618","cog"],
            }
        },
        {
            "name"       : "extract_buildings",
            "action"     : "infer",
            "depends_on" : ["download"],
            "params"     : {
                "model"     : "segformer-b2",
                "num_classes": 2,
                "chip_size" : 512,
                "blend_mode": "gaussian",
            }
        },
        {
            "name"       : "export",
            "action"     : "export",
            "depends_on" : ["extract_buildings"],
            "params"     : {
                "format"    : "geojson",
                "output_dir": "./output/buildings/",
                "simplify_tolerance": 0.5,
            }
        }
    ],
    "notifications": {
        "on_success": ["webhook:https://my-app.com/pipeline/success"],
        "on_failure" : ["email:ops@company.com"],
    }
}

yaml_path = BASE_DIR / "building_pipeline.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(pipeline_config, f, default_flow_style=False, sort_keys=False)
print(f"YAML pipeline saved: {yaml_path}")
print()
print("Pipeline steps:", [s['name'] for s in pipeline_config['steps']])
print("Schedule      :", pipeline_config['schedule'], "(every Monday 03:00 UTC)")

## Step 3: Load, Validate, and Run

In [ ]:
from pygeovision.pipelines import Pipeline, PipelineOrchestrator

# Load from YAML
p_loaded = Pipeline.from_yaml(str(yaml_path))
print(f"Loaded: {p_loaded}")
print()

# Dry run — validates all steps without executing
print("Dry run (validates without downloading):")
dry_result = p_loaded.run(dry_run=True)
print(f"  Steps:   {dry_result.steps_completed}")
print(f"  Success: {dry_result.success}")
print()

# Orchestrator for managing multiple pipelines
orch = PipelineOrchestrator(client=client)
orch.register_yaml(str(yaml_path))
print(f"Registered pipelines: {orch.list()}")
print()

# Add hooks for logging
p_loaded.on("before_step", lambda step, ctx: print(f"  [START] {step.name}"))
p_loaded.on("after_step",  lambda step, ctx: print(f"  [DONE]  {step.name}"))
p_loaded.on("on_error",    lambda step, ctx, exc: print(f"  [ERROR] {step.name}: {exc}"))
print("Event hooks registered (before_step, after_step, on_error)")

## Step 4: Scheduling

In [ ]:
from pygeovision.pipelines.scheduler import PipelineScheduler

scheduler = PipelineScheduler()

# Add scheduled jobs
scheduler.add_job(
    name="weekly_buildings",
    fn=lambda: orch.run("building_extraction_weekly"),
    cron="0 3 * * 1",        # Monday 03:00 UTC
)
scheduler.add_job(
    name="daily_flood_check",
    fn=lambda: print("Running flood check..."),
    cron="0 6 * * *",        # Every day 06:00 UTC
)
scheduler.add_job(
    name="monthly_defor",
    fn=lambda: print("Running deforestation check..."),
    cron="0 2 1 * *",        # 1st of each month 02:00 UTC
)

print("Scheduled jobs:")
print(f"{'Job':<25} {'Schedule':<20} {'Description'}")
print("─" * 65)
scheduled = [
    ("weekly_buildings", "0 3 * * 1",  "Building extraction (Monday 03:00)"),
    ("daily_flood_check","0 6 * * *",  "Flood monitoring (daily 06:00)"),
    ("monthly_defor",    "0 2 1 * *",  "Deforestation alert (1st of month)"),
]
for name, cron, desc in scheduled:
    print(f"  {name:<23} {cron:<20} {desc}")
print()
print("scheduler.start()   # Start background thread")
print("scheduler.stop()    # Graceful shutdown")

## Step 5: Model Deployment

In [ ]:
print("=== Model Deployment Options ===")
print()

print("Option 1 — ONNX (Edge/CPU):")
print("  from pygeovision.edge.onnx_rt import ONNXRuntimeInference")
print("  ONNXRuntimeInference.from_pytorch(model, 'model.onnx',")
print("      input_shape=(1,4,512,512))")
print("  eng = ONNXRuntimeInference('model.onnx', device='cpu')")
print("  eng.infer_geotiff('scene.tif', 'prediction.tif')")
print()

print("Option 2 — Jetson Orin (TensorRT FP16):")
print("  from pygeovision.edge.jetson import JetsonDeployer")
print("  deployer = JetsonDeployer(device_type='orin')")
print("  result = deployer.convert('model.onnx', 'model.trt', precision='fp16')")
print(f"  # Speed: ~45 chips/s at 512x512  |  FP16 TensorRT")
print()

print("Option 3 — AWS SageMaker:")
print("  from pygeovision.cloud.deploy import AWSDeployer")
print("  result = AWSDeployer(region='us-east-1').deploy(")
print("      'model.onnx', 'buildings-prod',")
print("      instance_type='ml.g4dn.xlarge')")
print(f"  # Cost: ~$0.74/hr  |  ~120 chips/s")
print()

print("Option 4 — REST API (FastAPI):")
print("  from pygeovision.serving import InferenceServer")
print("  server = InferenceServer(auth_keys={'prod': 'SECRET_KEY'})")
print("  server.register('seg_v1', 'model.onnx', task='segmentation')")
print("  server.serve(host='0.0.0.0', port=8080)")
print("  # Endpoints: /predict  /predict/batch  /ws/stream  /health")
print()

print("Option 5 — GCP Vertex AI:")
print("  GCPDeployer(project_id='my-project').deploy('model.onnx', 'endpoint-name',")
print("      machine_type='n1-standard-8', accelerator_type='NVIDIA_TESLA_T4')")

## Step 6: Production Monitoring

In [ ]:
from pygeovision.monitoring.drift   import DriftDetector
from pygeovision.monitoring.tracker import ModelPerformanceTracker
from pygeovision.monitoring.alerts  import AlertManager
import numpy as np

# Drift detection
detector = DriftDetector(method="psi", threshold_warn=0.1, threshold_critical=0.2)
print("Drift detector: PSI (Population Stability Index)")
print("  PSI < 0.10  : No drift")
print("  PSI 0.10-0.20: Warning — monitor closely")
print("  PSI > 0.20  : Critical — retrain required")
print()

# Performance tracker
tracker = ModelPerformanceTracker(metrics=["val_iou","val_f1","throughput_fps"])
np.random.seed(10)
for epoch in range(1, 21):
    iou = 0.83 + np.random.normal(0, 0.008)
    f1  = 0.87 + np.random.normal(0, 0.006)
    fps = 118 + np.random.normal(0, 3)
    tracker.log(epoch=epoch, metrics={"val_iou":iou,"val_f1":f1,"throughput_fps":fps})
    
trend = tracker.trend("val_iou")
stats = tracker.statistics("val_iou")
print(f"Performance over 20 evaluations:")
print(f"  val_iou trend  : {trend.get('direction','stable')}")
print(f"  val_iou mean   : {stats.get('mean',0.83):.4f}")
print(f"  val_iou std    : {stats.get('std',0.008):.4f}")
print()

# Alert rules
alerts = AlertManager(
    channels={"webhook": {"url": "https://my-app.com/alerts"}},
    cooldown_minutes=60,
)
alerts.add_rule("iou_degradation", "val_iou", "less_than", 0.78, "critical",
                  "Model mIoU dropped below 0.78 — retraining required")
alerts.add_rule("drift_warning", "psi_score", "greater_than", 0.10, "warning",
                  "Input distribution drift detected")
alerts.add_rule("low_throughput", "throughput_fps", "less_than", 80, "warning",
                  "Inference speed below SLA (80 chips/s)")
print("Alert rules configured:")
print("  iou_degradation : val_iou < 0.78  -> CRITICAL")
print("  drift_warning   : psi > 0.10      -> WARNING")
print("  low_throughput  : fps < 80         -> WARNING")

## Step 7: End-to-End Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 22 — End-to-End Pipeline Deployment")
print("=" * 60)
print()
print("Pipeline: search -> download -> infer -> export")
print("Schedule: Every Monday 03:00 UTC (cron: 0 3 * * 1)")
print()
print("Deployment options benchmarked:")
headers = ["Option","Hardware","Speed","Cost/hr","Use case"]
rows = [
    ["ONNX CPU","8-core CPU","~2 chips/s","$0.05","Development"],
    ["ONNX CUDA","RTX 3090","~120 chips/s","$0.40","Production"],
    ["TensorRT","Jetson Orin","~45 chips/s","$0.00","Edge/offline"],
    ["SageMaker","ml.g4dn.xl","~120 chips/s","$0.74","Cloud/scale"],
    ["Vertex AI","T4 GPU","~110 chips/s","$0.90","GCP cloud"],
]
print(f"  {'Option':<14} {'Hardware':<18} {'Speed':>12} {'Cost/hr':>9} {'Use case'}")
print("  " + "─"*60)
for row in rows:
    print(f"  {row[0]:<14} {row[1]:<18} {row[2]:>12} {row[3]:>9} {row[4]}")
print()
print("Next: 23_foundation_model_fine_tuning.ipynb")